# JackSparrow v43 multi-head — Delta Exchange India (Colab)

Train one **multi-head** bundle (`metadata_v43.json` + `model_artifact_v43.pkl`) with intraday horizons:

| Key | Forward bars | ~Minutes |
|-----|----------------|----------|
| scalp_10m | 2 | 10m |
| intraday_30m | 6 | 30m |
| trend_1h | 12 | 1h |
| swing_2h | 24 | 2h |

Local export: `python scripts/train_v43_multihead_export.py --feature-csv <path>`


## References (in your cloned repo)

- `docs/03-ml-models.md` — model storage and v43 path
- `docs/feature_entrypoints_audit.md` — train/serve entrypoints
- `feature_store/jacksparrow_v43_contract.py` — feature order + version gate
- `feature_store/jacksparrow_v43_build_matrix.py` — `build_v43_feature_matrix`
- `agent/models/v43_pickle_shims.py` — `EnsembleModel`, `LGBMModel`, `FeatureEngineer` pickle contract


In [21]:
# Minimal Colab dependencies (pin versions in prod if needed)
%pip install -q pandas numpy requests joblib scikit-learn lightgbm xgboost


## 1) Repository on `sys.path`

**Preferred:** just run the next cell — it clones the **`major-rework`** branch automatically and switches to it if an older clone is already present.

Manual equivalent:

```bash
git clone --branch major-rework https://github.com/energyforreal/JackSparrow.git /content/trading-agent
```

If you omit the target directory, Git creates `/content/JackSparrow` — the next cell detects that too.

(Private repo: use a [personal access token](https://github.com/settings/tokens) or SSH URL instead.)

This notebook needs **`feature_store/jacksparrow_v43_build_matrix.py`** and **`jacksparrow_v43_contract.py`** on the branch you clone. If the repo-root cell fails after `git pull`, push those files from your local checkout.

Alternatively mount Google Drive containing a checkout and set environment variable `TRADING_AGENT_ROOT`
to that folder before importing.


In [ ]:
import subprocess
from pathlib import Path

_CLONE = Path("/content/trading-agent")
_URL = "https://github.com/energyforreal/JackSparrow.git"
_BRANCH = "major-rework"

if (_CLONE / ".git").is_dir():
    print("Repo already present:", _CLONE)
    result = subprocess.run(
        ["git", "-C", str(_CLONE), "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True,
    )
    current_branch = result.stdout.strip()
    if current_branch != _BRANCH:
        print(f"Switching from '{current_branch}' to '{_BRANCH}'...")
        subprocess.run(["git", "-C", str(_CLONE), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "checkout", _BRANCH], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    else:
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    print(f"On branch '{_BRANCH}', up to date.")
else:
    subprocess.run(["git", "clone", "--branch", _BRANCH, _URL, str(_CLONE)], check=True)
    print(f"Cloned branch '{_BRANCH}' to", _CLONE)


## 1b) If the repo-root cell fails: wrong branch cloned

Diagnostics showing **`/content/trading-agent/.../jacksparrow_v43_build_matrix.py` exists: False** mean Colab cloned the **wrong branch** — the v43 modules only exist on `major-rework`, not `main`.

**Fix A — re-run the §1 clone cell (handles this automatically).**

The §1 clone cell now always clones / switches to `major-rework`. Simply re-run it, then re-run this repo-root detection cell.

If you need to fix it manually in Colab (new code cell):

```python
import subprocess
subprocess.run(["git", "-C", "/content/trading-agent", "fetch", "origin"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "checkout", "major-rework"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "pull", "origin", "major-rework"], check=True)
print("Done — now re-run from the repo-root detection cell")
```

**Fix B (no-GitHub upload):** run the **next** code cell once, upload the required `.py` files from your PC’s `feature_store/` folder, then **re-run** the repo-root detection cell.


In [ ]:
# Colab only: upload v43 modules if they are not on GitHub yet.
# Set RUN_V43_UPLOAD = True, run this cell, pick the two files from your PC, then re-run the repo-root cell.

RUN_V43_UPLOAD = False

from pathlib import Path

if not RUN_V43_UPLOAD:
    print("Skipping upload (set RUN_V43_UPLOAD = True to upload from PC).")
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload path only works on Google Colab. Use git push on your PC.") from exc

    dest = Path("/content/trading-agent/feature_store")
    dest.mkdir(parents=True, exist_ok=True)
    print("Upload: jacksparrow_v43_build_matrix.py and jacksparrow_v43_contract.py")
    for name, data in files.upload().items():
        base = Path(name).name
        if not base.lower().endswith(".py"):
            continue
        out = dest / base
        out.write_bytes(data)
        print("Wrote", out, "(", out.stat().st_size, "bytes)")
    print("Done. Re-run the repo-root cell above, then continue.")


In [ ]:
import os
import sys
from pathlib import Path

# Colab: uncomment if your clone is not auto-detected
# os.environ["TRADING_AGENT_ROOT"] = "/content/trading-agent"

_MARKER = Path("feature_store") / "jacksparrow_v43_build_matrix.py"

def _repo_has_marker(c: Path) -> bool:
    try:
        return (c / _MARKER).is_file()
    except OSError:
        return False


candidates: list[Path] = []
if os.environ.get("TRADING_AGENT_ROOT"):
    candidates.append(Path(os.environ["TRADING_AGENT_ROOT"]).resolve())
# Default clone targets (see section 1). `git clone <url>` alone creates /content/<repo_name>/.
for fixed in ("/content/trading-agent", "/content/JackSparrow", "/content/jacksparrow"):
    candidates.append(Path(fixed).resolve())
candidates.append(Path.cwd().resolve())
p = Path.cwd().resolve()
for _ in range(16):
    candidates.append(p)
    if p == p.parent:
        break
    p = p.parent
_content = Path("/content")
if _content.is_dir():
    try:
        _skip_names = frozenset({"sample_data", ".config"})
        for child in sorted(_content.iterdir()):
            if child.is_dir() and not child.name.startswith(".") and child.name not in _skip_names:
                candidates.append(child.resolve())
    except OSError:
        pass

seen: set[Path] = set()
deduped: list[Path] = []
for c in candidates:
    try:
        r = c.resolve()
    except OSError:
        continue
    if r not in seen:
        seen.add(r)
        deduped.append(r)

REPO_ROOT = None
for c in deduped:
    if _repo_has_marker(c):
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    listing = ""
    if _content.is_dir():
        try:
            names = sorted(
                p.name for p in _content.iterdir() if p.is_dir() and not p.name.startswith(".")
            )
            listing = f" Top-level folders under /content: {names!r}."
        except OSError:
            pass
    print(f"Diagnosing: Checking for {_MARKER} in detected candidates.")
    for c in deduped:
        expected_path = c / _MARKER
        print(f"  Checking: {expected_path} (exists: {expected_path.exists()}, is_file: {expected_path.is_file()})")
    raise FileNotFoundError(
        f"Could not find repo root (need {_MARKER.as_posix()}). "
        "Push that file to GitHub (or merge branch that has it), then `git pull` under your clone, "
        "or set TRADING_AGENT_ROOT to a checkout that contains feature_store/. See notebook section 1b (git push or Colab upload)."
        + listing
    )

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
# Optional: verify v43 training modules exist on REPO_ROOT before heavy downloads
from pathlib import Path

_fs = Path(REPO_ROOT) / "feature_store"
for name in (
    "jacksparrow_v43_build_matrix.py",
    "jacksparrow_v43_contract.py",
    "jacksparrow_v43_train_multihead.py",
    "jacksparrow_v43_multihead.py",
):
    p = _fs / name
    print(name, "OK" if p.is_file() else "MISSING — push from your PC then git pull in Colab")


In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import time
import warnings
from datetime import datetime, timezone

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor

from feature_store.jacksparrow_v43_build_matrix import build_v43_feature_matrix
from feature_store.jacksparrow_v43_contract import (
    V43_CANONICAL_FEATURES,
    V43_COMPATIBLE_FEATURE_VERSION,
    validate_v43_metadata_compatibility,
)
from feature_store.jacksparrow_v43_multihead import (
    V43_MULTIHEAD_ARTIFACT_FORMAT,
    V43_MULTIHEAD_MODEL_FAMILY,
)
from feature_store.jacksparrow_v43_train_multihead import (
    artifact_dict_from_bundle,
    train_multihead_from_feature_matrix,
)
from agent.models.v43_pickle_shims import (
    EnsembleModel,
    FeatureEngineer,
    LGBMModel,
    MultiHeadBundle,
    set_v43_build_feature_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)



## 2) Delta Exchange India — historical candles

Public `GET /v2/history/candles` on `https://api.india.delta.exchange` (same as
`agent/data/historical_data_loader.py`). Use `page_size` ≤ 2000 and polite delays between calls.

Default pull: **160,000** five-minute candles (2× prior 80k target) and matching funding lookback.
Large pulls can take **10–25 minutes** on Colab depending on rate limits.


In [ ]:
DELTA_BASE = os.environ.get("DELTA_EXCHANGE_BASE_URL", "https://api.india.delta.exchange")
SYMBOL = os.environ.get("DELTA_SYMBOL", "BTCUSD")
RESOLUTION_5M = "5m"

TARGET_CANDLES_5M = int(os.environ.get("TARGET_CANDLES_5M", "160000"))
FUNDING_LOOKBACK_HOURS = int(os.environ.get("FUNDING_LOOKBACK_HOURS", "16000"))

REQUEST_DELAY_S = float(os.environ.get("DELTA_REQUEST_DELAY_S", "0.35"))
PAGE_SIZE = 2000

RESOLUTION_SECONDS = {"5m": 300, "15m": 900, "30m": 1800, "1h": 3600, "2h": 7200}


def _ts_col(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in ("open", "high", "low", "close", "volume"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["timestamp"] = pd.to_datetime(out["time"], unit="s", utc=True)
    out = out.drop(columns=["time"], errors="ignore")
    return out.sort_values("timestamp").reset_index(drop=True)


def fetch_candles_range(session, symbol, resolution, start_ts, end_ts, max_retries=4):
    url = f"{DELTA_BASE}/v2/history/candles"
    params = {
        "symbol": symbol,
        "resolution": resolution,
        "start": int(start_ts),
        "end": int(end_ts),
        "page_size": PAGE_SIZE,
    }
    last_exc: Exception = RuntimeError("no attempts made")
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if not data.get("success"):
                raise RuntimeError(data)
            rows = data.get("result") or []
            return pd.DataFrame(rows)
        except (requests.exceptions.RequestException, RuntimeError) as exc:
            last_exc = exc
            if attempt < max_retries - 1:
                backoff = 2 ** attempt
                print(f"  fetch_candles_range retry {attempt + 1}/{max_retries - 1} in {backoff}s: {exc}")
                time.sleep(backoff)
    raise last_exc


def fetch_5m_history(symbol, target_rows, resolution="5m"):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    sec = RESOLUTION_SECONDS[resolution]
    end_ts = int(time.time())
    span = int(target_rows * sec * 1.15)
    start_ts = end_ts - span
    chunk_sec = PAGE_SIZE * sec
    est_chunks = max(1, int((end_ts - start_ts) / chunk_sec))
    print(f"  target_rows={target_rows:,} (~{target_rows * sec / 86400:.0f} days), est. API chunks: {est_chunks}")
    frames = []
    cursor = start_ts
    chunk_i = 0
    while cursor < end_ts:
        chunk_i += 1
        chunk_end = min(end_ts, cursor + chunk_sec)
        if chunk_i == 1 or chunk_i % 25 == 0 or chunk_end >= end_ts:
            print(f"  chunk {chunk_i}/{est_chunks} ...")
        df = fetch_candles_range(session, symbol, resolution, cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    raw = raw.drop_duplicates(subset=["time"]).sort_values("time")
    if len(raw) > target_rows:
        raw = raw.iloc[-target_rows:]
    return _ts_col(raw)


def fetch_funding_hourly(symbol, lookback_hours):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    end_ts = int(time.time())
    start_ts = end_ts - int(lookback_hours * 3600)
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        chunk_end = min(end_ts, cursor + PAGE_SIZE * 3600)
        df = fetch_candles_range(session, f"FUNDING:{symbol}", "1h", cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame(columns=["timestamp", "funding_rate"])
    raw = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["time"]).sort_values("time")
    raw["timestamp"] = pd.to_datetime(raw["time"], unit="s", utc=True)
    raw["funding_rate"] = pd.to_numeric(raw["close"], errors="coerce")
    return raw[["timestamp", "funding_rate"]].reset_index(drop=True)


print("Fetching 5m OHLCV ...")
df_5m = fetch_5m_history(SYMBOL, TARGET_CANDLES_5M, RESOLUTION_5M)
print("5m rows:", len(df_5m), "range:", df_5m["timestamp"].min(), "->", df_5m["timestamp"].max())

print("Fetching funding (1h) ...")
try:
    df_funding = fetch_funding_hourly(SYMBOL, FUNDING_LOOKBACK_HOURS)
    print("funding rows:", len(df_funding))
except Exception as _fund_exc:
    print(f"WARNING: funding fetch failed ({_fund_exc!r}). Continuing with empty funding data.")
    df_funding = pd.DataFrame(columns=["timestamp", "funding_rate"])



## 3) Features and multi-head labels (v43 contract)

- Features: `build_v43_feature_matrix(...)` then `V43_CANONICAL_FEATURES` order.
- Labels: per-head simple forward returns via `train_multihead_from_feature_matrix` (2/6/12/24 bars).
- Export: single `MultiHeadBundle` artifact + `horizons{}` metadata (see §5).


In [ ]:
df_feat = build_v43_feature_matrix(
    df_5m,
    None,
    None,
    df_funding,
    for_training=True,
    primary_interval="5m",
)

feat_cols = list(V43_CANONICAL_FEATURES)
if "timestamp" in df_feat.columns:
    _label_idx = pd.to_datetime(df_feat["timestamp"], utc=True)
else:
    _label_idx = pd.to_datetime(df_5m["timestamp"], utc=True).iloc[: len(df_feat)]
_close_5m = pd.Series(
    pd.to_numeric(df_5m["close"], errors="coerce").values,
    index=pd.to_datetime(df_5m["timestamp"], utc=True),
)
close = _close_5m.reindex(_label_idx)

_maker_fee = float(os.environ.get("V43_MAKER_FEE", "0.0005"))
_slippage = float(os.environ.get("V43_SLIPPAGE", "0.0003"))
_leverage = int(os.environ.get("V43_LEVERAGE", "3"))
_train_rng = int(os.environ.get("V43_TRAIN_RNG", "42"))

multi_bundle, metadata = train_multihead_from_feature_matrix(
    df_feat[feat_cols],
    close,
    feat_cols=feat_cols,
    validation_fraction=float(os.environ.get("V43_VALIDATION_FRACTION", "0.15")),
    rng=_train_rng,
    maker_fee=_maker_fee,
    slippage=_slippage,
    leverage=_leverage,
    cost_aware_labels=True,
)

validation_metrics = metadata["horizons"]["intraday_30m"]["validation_metrics"]
split_metadata = metadata.get("split", {})
primary_ensemble = multi_bundle.get_head(6)
print("Multi-head training complete. Horizons:", list(metadata.get("horizons", {}).keys()))
print("Primary execution head (6 bars) validation_corr:", validation_metrics.get("validation_corr"))
print("round_trip_cost_pct:", metadata.get("runtime_cost_assumptions", {}).get("round_trip_cost_pct"))


In [ ]:
from feature_store.jacksparrow_v43_multihead import format_horizon_training_diagnostics

print(format_horizon_training_diagnostics(metadata))


## 4) Legacy single-head cells (skipped)

Per-head training, meta-stack, and walk-forward analysis now run inside
`feature_store/jacksparrow_v43_train_multihead.py` for all four horizons in one export.


In [ ]:
# Legacy single-head training removed — see multi-head train cell above.


## 4c) Meta-learner stacking (Pattern 3)

Base regressors (LGBM + XGB + RF) are unchanged. A **LGBMClassifier** meta-learner maps the
stacked regressor outputs to direction probability; a **Ridge** calibrator maps probability
back to expected-return scale so gate-5 edge-vs-cost math stays valid at runtime.

Stored on ``ensemble.meta`` and ``ensemble.calibrator`` — routed automatically by
``EnsembleModel.predict()`` in ``v43_pickle_shims.py``.

Meta stacking (LGBM classifier + Ridge calibrator) runs **inside** ``train_multihead_from_feature_matrix``
for each horizon. The cells below are stubs — no separate §4c step is required before export.


In [ ]:
# Legacy single-head training removed — see multi-head train cell above.


In [ ]:
# Legacy single-head training removed — see multi-head train cell above.


## 5) `FeatureEngineer` + export

`FeatureEngineer` from `v43_pickle_shims` stores only the canonical column contract. The actual transform path is registered with `set_v43_build_feature_matrix(...)`, matching the agent startup path and avoiding fragile notebook-local closure pickling.


## Export gates (production vs research)

- **Production (Colab default)**: `V43_EXPORT_GATES_STRICT=true` + **primary-only** — strict mins on `scalp_10m` + `intraday_30m` only; `trend_1h`/`swing_2h` need meta_auc ≥ **0.50**.
- **Full strict (all four heads)**: `os.environ["V43_EXPORT_STRICT_PRIMARY_ONLY"] = "false"`.
- **Research relaxed**: `V43_EXPORT_GATES_STRICT=false`.
- **Do not promote** bundles to `agent/model_storage/` unless strict gates pass without env overrides.

Default history: **160,000** × 5m bars (~556 days) + **16,000** funding hours. Override with `TARGET_CANDLES_5M` / `FUNDING_LOOKBACK_HOURS` if needed.


In [ ]:
def make_feature_engineer():
    fe = FeatureEngineer()
    fe.columns = list(V43_CANONICAL_FEATURES)
    return fe


def _version_or_none(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def _git_commit_or_none():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=str(REPO_ROOT),
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
    except Exception:
        return None


set_v43_build_feature_matrix(build_v43_feature_matrix)
fe = make_feature_engineer()

OUT_DIR = Path(os.environ.get("V43_EXPORT_DIR", "jacksparrow_v43_bundle"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = OUT_DIR / "model_artifact_v43.pkl"
metadata_path = OUT_DIR / "metadata_v43.json"

assert validation_metrics.get("inference_path") == "meta_calibrator", (
    "train_multihead_from_feature_matrix must produce meta_calibrator validation_metrics."
)
from feature_store.jacksparrow_v43_multihead import (
    format_export_gate_summary,
    resolve_min_meta_auc_by_horizon,
    validate_multihead_export_gates,
)

# Export gates: strict production mins, or relaxed (warn-only above hard floor 0.52).
# Colab: os.environ["V43_EXPORT_GATES_STRICT"] = "false"  # allow near-miss heads
# Override mins: V43_MIN_META_AUC_INTRADAY_30M=0.55 etc.
_export_strict = os.environ.get("V43_EXPORT_GATES_STRICT", "true").strip().lower() in (
    "1", "true", "yes",
)
print(format_export_gate_summary(metadata, strict_primary_only=True))
print(f"V43_EXPORT_GATES_STRICT={_export_strict}")
print("V43_EXPORT_STRICT_PRIMARY_ONLY=True (1h/2h use secondary floor 0.50)")
gate_failures, gate_warnings = validate_multihead_export_gates(
    metadata,
    strict=_export_strict,
    return_soft=True,
    strict_primary_only=True,
)
for _gw in gate_warnings:
    print(f"EXPORT WARN: {_gw}")
if gate_failures:
    for _gf in gate_failures:
        print(f"EXPORT GATE: {_gf}")
    raise AssertionError("Export blocked: " + "; ".join(gate_failures))
if gate_warnings and not _export_strict:
    print("Export allowed with warnings (set V43_EXPORT_GATES_STRICT=true for production).")

_mins = resolve_min_meta_auc_by_horizon()
primary_vm = validation_metrics
print(
    f"Primary horizon intraday_30m: meta_auc={primary_vm.get('meta_auc'):.4f} "
    f"(min {_mins['intraday_30m']:.2f}), "
    f"corr={primary_vm.get('validation_corr'):.4f}"
)

# Hit-rate gates: gross >50% is desirable but not a hard block at low-AUC signal levels.
# Model is viable as long as corr > 0 and meta_auc > 0.50; hit-rate warnings are informational.
_long_hr = validation_metrics["long_candidates"].get("hit_rate_gross_positive") or 0.0
_short_hr = validation_metrics["short_candidates"].get("hit_rate_gross_positive") or 0.0
if _long_hr < 0.50:
    print(f"WARNING: long hit rate (gross) = {_long_hr:.4f} < 0.50 — "
          "model is directionally weak on long candidates; review before live deployment.")
if _short_hr < 0.50:
    print(f"WARNING: short hit rate (gross) = {_short_hr:.4f} < 0.50 — "
          "model is directionally weak on short candidates; review before live deployment.")

artifact_payload = artifact_dict_from_bundle(multi_bundle, fe)
artifact_payload.update({
    "feature_engineer": fe,
    "features": list(V43_CANONICAL_FEATURES),
    })

joblib.dump(artifact_payload, artifact_path)

_agent_load_snippet = "\n".join(
    [
        "artifact = joblib.load(MODEL_ARTIFACT_PATH)",
        "model    = artifact[\"model\"]",
        "features = artifact[\"features\"]",
        "fe       = artifact[\"feature_engineer\"]",
        "df_feat  = fe.transform(df5, df15, df1h, df_funding, include_target=False)",
        "assert len(df_feat) >= 2, \"Need at least 2 rows to use the last closed bar\"",
        "X        = df_feat[features].iloc[[-2]].values",
        "X_df     = df_feat[features].iloc[[-2]]",
        "expected_return = model.predict_head(6, X, X_df=X_df)  # primary 30m head",
    ]
)

meta = dict(metadata)
meta.update({
    "version": "v43",
    "compatible_feature_version": V43_COMPATIBLE_FEATURE_VERSION,
    "symbol": SYMBOL,
    "model_name": "jacksparrow_v43_" + str(SYMBOL),
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "feature_count": len(V43_CANONICAL_FEATURES),
    "features": list(V43_CANONICAL_FEATURES),
    "model_family": V43_MULTIHEAD_MODEL_FAMILY,
    "artifact_format": V43_MULTIHEAD_ARTIFACT_FORMAT,
    "primary_execution_horizon_bars": 6,
    "target_definition": metadata.get("target_definition", "cost_aware_forward_return"),
    "model_architecture": {
        "base_estimators": ["lgbm_regressor", "xgb_regressor", "rf_regressor"],
        "meta_learner": "lgbm_classifier" if getattr(primary_ensemble, "meta", None) is not None else None,
        "calibrator": "ridge" if getattr(primary_ensemble, "calibrator", None) is not None else None,
    },
    "split": split_metadata,
    "data": {
        "delta_base_url": DELTA_BASE,
        "symbol": SYMBOL,
        "resolution": RESOLUTION_5M,
        "target_candles_5m": TARGET_CANDLES_5M,
        "funding_lookback_hours": FUNDING_LOOKBACK_HOURS,
        "candles_5m_rows": int(len(df_5m)),
        "funding_rows": int(len(df_funding)),
        "candles_5m_start": df_5m["timestamp"].min().isoformat() if len(df_5m) else None,
        "candles_5m_end": df_5m["timestamp"].max().isoformat() if len(df_5m) else None,
    },
    "provenance": {
        "repo_root": str(REPO_ROOT),
        "git_commit": _git_commit_or_none(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            "joblib": _version_or_none("joblib"),
            "numpy": _version_or_none("numpy"),
            "pandas": _version_or_none("pandas"),
            "scikit-learn": _version_or_none("scikit-learn"),
            "lightgbm": _version_or_none("lightgbm"),
            "xgboost": _version_or_none("xgboost"),
        },
    },
    "agent_load": _agent_load_snippet,
})
metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", artifact_path)
print("Wrote", metadata_path)



## 6) Alignment self-check + handoff

Run `validate_v43_metadata_compatibility` on the exported JSON. Then copy the bundle folder into
`agent/model_storage/` (for example `JackSparrow_v43_models_BTCUSD/`) and configure model discovery per
`docs/03-ml-models.md`.

**On your machine (after download):**

```text
pytest tests/unit/test_jacksparrow_v43_contract.py
python scripts/smoke_test_v43.py
pytest tests/unit/test_jack_sparrow_v43_node_ctx.py
```


In [ ]:
with metadata_path.open(encoding="utf-8") as f:
    meta_loaded = json.load(f)
validate_v43_metadata_compatibility(meta_loaded)

set_v43_build_feature_matrix(build_v43_feature_matrix)
tail5 = df_5m.tail(800).copy()
sm = fe.transform(tail5, None, None, df_funding, include_target=False)
assert len(sm) >= 2
assert list(sm.columns[: len(V43_CANONICAL_FEATURES)]) == list(V43_CANONICAL_FEATURES)
_feat_list = list(V43_CANONICAL_FEATURES)
expected_return = multi_bundle.get_head(6).predict(
    sm[_feat_list].iloc[[-2]].values,
    X_df=sm[_feat_list].iloc[[-2]],
)
assert np.atleast_1d(np.asarray(expected_return, dtype=np.float64)).size == 1
assert "horizons" in meta_loaded and "intraday_30m" in meta_loaded["horizons"]
assert "validation_metrics" in meta_loaded["horizons"]["intraday_30m"]
assert "split" in meta_loaded
print("validate_v43_metadata_compatibility: OK")
print("metadata split + validation metrics: OK")
print("fe.transform + multi_bundle.get_head(6).predict smoke: OK", expected_return)



In [ ]:
try:
    from google.colab import files
    import shutil

    z = Path("/content/jacksparrow_v43_bundle.zip")
    shutil.make_archive(str(z.with_suffix("")), "zip", root_dir=str(OUT_DIR))
    files.download(str(z))
except Exception as e:
    print("Colab zip/download skipped:", e)

